# MB25 — Reservas de material — Análise Exploratória

**Tabela:** `prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material`
**Ambiente:** 🔴 **PRODUÇÃO (PRD)**
**Transação SAP:** MB25 · **Objeto:** **VIEW**
**Colunas:** 49 · **Clustering:** _(não se aplica — é view)_

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** e **somente leitura** — nenhuma cria,
altera ou grava objeto. Não há widget, view temporária nem ordem obrigatória.

## ⚠️ Cuidados por ser PRODUÇÃO

- **Somente consultas `SELECT` / `DESCRIBE`** — nenhum `CREATE`, `INSERT` ou gravação neste notebook.
- **Volume real de produção:** várias células varrem a view inteira. Rode fora do horário de pico
  se o cluster for compartilhado.
- **Dados pessoais:** `nm_usuario` e `nm_recebedor_mercadoria` identificam pessoas. Por precaução,
  este notebook as **mascara por hash** nas amostras e as analisa só por contagem.

## Alertas de partida _(extraídos dos comentários do `DESCRIBE`)_

| Severidade | Alerta |
|---|---|
| 🔴 ALTA | **`qt_saldo_reserva` é calculado** (`qt_necessaria − qt_retirada`). A seção 18.1 valida a fórmula linha a linha. |
| 🔴 ALTA | **7 campos de cabeçalho replicados em todos os itens** (imobilizado, subimobilizado, diagrama de rede, ordem do cliente e afins). Na RESB são campos de item — aqui o valor do cabeçalho foi copiado. Itens com valor próprio no SAP vão **divergir** (seção 18.3). |
| 🟡 MÉDIA | **3 datas em `string`** (`dt_necessidade`, `dt_reserva`, `dt_criacao`) — normalizar antes de comparar. |
| 🟡 MÉDIA | **5 indicadores `ind_` em STRING** (`X` / vazio), não boolean — mesmo padrão do SAP, comparação direta. |
| 🟡 MÉDIA | **`dt_criacao` vem de `system_date`**, não de um campo de negócio da reserva — pode não bater com a data de criação exibida na MB25. |
| ⚪ BAIXA | **Tabela semelhante em DEV:** `dev_procurement.corp_curated.tbl_ds_pro_zglmm368` (ZGLMM368) também é de reservas — os achados de lá ajudam a interpretar os daqui. |

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados |
| 2 | Volumetria |
| 3 | Distribuição por centro — escolha do cenário |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| **6** | **Preenchimento de todas as colunas** |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 9 | Indicadores `X` / vazio |
| 10 | Perfil numérico |
| **11** | **Totais para conciliação com o SAP** |
| 12 | Datas |
| 13 | Códigos e zeros à esquerda |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| 17 | Freshness |
| **18** | **Análises específicas da MB25** |
| 19 | Amostra ampliada |
| 20 | Resumo |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 14.


## 1. Metadados

Objeto é **view** — `DESCRIBE DETAIL` e `DESCRIBE HISTORY` não se aplicam e ficaram comentados
para não interromper o **Run All**.

In [0]:
-- 1. ESTRUTURA
DESCRIBE EXTENDED prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material;

In [0]:
-- 1.1 Detalhe e historico nao se aplicam a view.
-- DESCRIBE DETAIL prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material;
-- DESCRIBE HISTORY prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material LIMIT 20;
SELECT 'Objeto e uma VIEW — os dois comandos acima nao se aplicam' AS observacao;

## 2. Volumetria

In [0]:
-- 2. VOLUMETRIA
SELECT COUNT(*)                                            AS linhas,
       COUNT(DISTINCT num_reserva)                         AS reservas,
       COUNT(DISTINCT CONCAT_WS('|', num_reserva, num_item_reserva)) AS itens_de_reserva,
       COUNT(DISTINCT cod_centro)                          AS centros,
       COUNT(DISTINCT cod_material)                        AS materiais,
       COUNT(DISTINCT num_ordem)                           AS ordens
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material;

## 3. Distribuição por centro

Base para escolher o cenário de teste: recorte com volume viável (10 mil a 300 mil linhas).
Vale conferir se os centros 4128 e 4014 — usados nos demais testes — aparecem na faixa.

In [0]:
-- 3. DISTRIBUICAO POR CENTRO
SELECT COALESCE(NULLIF(trim(cod_centro), ''), '(vazio)') AS cod_centro,
       COUNT(*)                        AS linhas,
       COUNT(DISTINCT num_reserva)     AS reservas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
GROUP BY COALESCE(NULLIF(trim(cod_centro), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 60;

In [0]:
-- 3.1 CENTROS JA USADOS NOS DEMAIS TESTES
SELECT cod_centro,
       COUNT(*)                    AS linhas,
       COUNT(DISTINCT num_reserva) AS reservas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material), 2) AS pct
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
WHERE cod_centro IN ('4128', '4014')
GROUP BY cod_centro
ORDER BY cod_centro;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional.

No SAP, a RESB é identificada por **reserva + item + tipo de registro** (RSNUM + RSPOS + RSART).
A seção confirma se a view preserva esse grão.

In [0]:
-- 4. GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material),
g AS (
  SELECT 'num_reserva' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_reserva` FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material)
UNION ALL
  SELECT 'num_reserva + num_item_reserva' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva` FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material)
UNION ALL
  SELECT 'num_reserva + num_item_reserva + tp_registro' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva`, `tp_registro` FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material)
UNION ALL
  SELECT 'num_reserva + num_item_reserva + tp_registro + dateingest' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva`, `tp_registro`, `dateingest` FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material)
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `num_reserva + num_item_reserva`.

**Regra:** linhas idênticas = duplicata real. Linhas distintas = granularidade legítima
(tipo de registro, ou acúmulo de cargas se a view for histórica).

In [0]:
-- 5. CHAVES DUPLICADAS
SELECT `num_reserva`, `num_item_reserva`, COUNT(*) AS qtd
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
GROUP BY `num_reserva`, `num_item_reserva`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [0]:
-- 5.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS
-- Colunas pessoais excluidas: nm_usuario, nm_recebedor_mercadoria
WITH dup AS (
  SELECT `num_reserva`, `num_item_reserva` FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `num_reserva`, `num_item_reserva` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material c JOIN dup ON c.`num_reserva` <=> dup.`num_reserva` AND c.`num_item_reserva` <=> dup.`num_item_reserva`
),
agg AS (
  SELECT `num_reserva`, `num_item_reserva`,
         COUNT(DISTINCT `tp_registro`) AS `tp_registro`,
         COUNT(DISTINCT `cod_material`) AS `cod_material`,
         COUNT(DISTINCT `desc_material`) AS `desc_material`,
         COUNT(DISTINCT `num_lote`) AS `num_lote`,
         COUNT(DISTINCT `cod_centro`) AS `cod_centro`,
         COUNT(DISTINCT `cod_deposito`) AS `cod_deposito`,
         COUNT(DISTINCT `cod_centro_receptor`) AS `cod_centro_receptor`,
         COUNT(DISTINCT `cod_deposito_receptor`) AS `cod_deposito_receptor`,
         COUNT(DISTINCT `dt_necessidade`) AS `dt_necessidade`,
         COUNT(DISTINCT `dt_reserva`) AS `dt_reserva`,
         COUNT(DISTINCT `dt_criacao`) AS `dt_criacao`,
         COUNT(DISTINCT `qt_necessaria`) AS `qt_necessaria`,
         COUNT(DISTINCT `qt_retirada`) AS `qt_retirada`,
         COUNT(DISTINCT `qt_saldo_reserva`) AS `qt_saldo_reserva`,
         COUNT(DISTINCT `sg_unidade_medida_basica`) AS `sg_unidade_medida_basica`,
         COUNT(DISTINCT `qt_um_registro`) AS `qt_um_registro`,
         COUNT(DISTINCT `sg_unidade_medida_registro`) AS `sg_unidade_medida_registro`,
         COUNT(DISTINCT `qt_verificacao_disponibilidade`) AS `qt_verificacao_disponibilidade`,
         COUNT(DISTINCT `num_ordem`) AS `num_ordem`,
         COUNT(DISTINCT `cod_operacao`) AS `cod_operacao`,
         COUNT(DISTINCT `tp_movimento`) AS `tp_movimento`,
         COUNT(DISTINCT `tp_necessidade`) AS `tp_necessidade`,
         COUNT(DISTINCT `cod_debito_credito`) AS `cod_debito_credito`,
         COUNT(DISTINCT `num_ordem_referencia`) AS `num_ordem_referencia`,
         COUNT(DISTINCT `num_ordem_servico`) AS `num_ordem_servico`,
         COUNT(DISTINCT `num_item_ordem_servico`) AS `num_item_ordem_servico`,
         COUNT(DISTINCT `st_reserva`) AS `st_reserva`,
         COUNT(DISTINCT `ind_movimento_permitido`) AS `ind_movimento_permitido`,
         COUNT(DISTINCT `ind_item_eliminado`) AS `ind_item_eliminado`,
         COUNT(DISTINCT `ind_registro_final`) AS `ind_registro_final`,
         COUNT(DISTINCT `ind_item_dummy`) AS `ind_item_dummy`,
         COUNT(DISTINCT `ind_material_granel`) AS `ind_material_granel`,
         COUNT(DISTINCT `cod_estoque_especial`) AS `cod_estoque_especial`,
         COUNT(DISTINCT `cod_centro_custo`) AS `cod_centro_custo`,
         COUNT(DISTINCT `cod_elemento_pep`) AS `cod_elemento_pep`,
         COUNT(DISTINCT `num_imobilizado`) AS `num_imobilizado`,
         COUNT(DISTINCT `num_sub_imobilizado`) AS `num_sub_imobilizado`,
         COUNT(DISTINCT `num_diagrama_rede`) AS `num_diagrama_rede`,
         COUNT(DISTINCT `num_ordem_cliente`) AS `num_ordem_cliente`,
         COUNT(DISTINCT `num_item_ordem_cliente`) AS `num_item_ordem_cliente`,
         COUNT(DISTINCT `num_divisao_ordem_cliente`) AS `num_divisao_ordem_cliente`,
         COUNT(DISTINCT `desc_observacao`) AS `desc_observacao`,
         COUNT(DISTINCT `dateingest`) AS `dateingest`,
         COUNT(DISTINCT `yearingest`) AS `yearingest`,
         COUNT(DISTINCT `monthingest`) AS `monthingest`
  FROM d GROUP BY `num_reserva`, `num_item_reserva`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(45,
    'tp_registro', MAX(`tp_registro`),
    'cod_material', MAX(`cod_material`),
    'desc_material', MAX(`desc_material`),
    'num_lote', MAX(`num_lote`),
    'cod_centro', MAX(`cod_centro`),
    'cod_deposito', MAX(`cod_deposito`),
    'cod_centro_receptor', MAX(`cod_centro_receptor`),
    'cod_deposito_receptor', MAX(`cod_deposito_receptor`),
    'dt_necessidade', MAX(`dt_necessidade`),
    'dt_reserva', MAX(`dt_reserva`),
    'dt_criacao', MAX(`dt_criacao`),
    'qt_necessaria', MAX(`qt_necessaria`),
    'qt_retirada', MAX(`qt_retirada`),
    'qt_saldo_reserva', MAX(`qt_saldo_reserva`),
    'sg_unidade_medida_basica', MAX(`sg_unidade_medida_basica`),
    'qt_um_registro', MAX(`qt_um_registro`),
    'sg_unidade_medida_registro', MAX(`sg_unidade_medida_registro`),
    'qt_verificacao_disponibilidade', MAX(`qt_verificacao_disponibilidade`),
    'num_ordem', MAX(`num_ordem`),
    'cod_operacao', MAX(`cod_operacao`),
    'tp_movimento', MAX(`tp_movimento`),
    'tp_necessidade', MAX(`tp_necessidade`),
    'cod_debito_credito', MAX(`cod_debito_credito`),
    'num_ordem_referencia', MAX(`num_ordem_referencia`),
    'num_ordem_servico', MAX(`num_ordem_servico`),
    'num_item_ordem_servico', MAX(`num_item_ordem_servico`),
    'st_reserva', MAX(`st_reserva`),
    'ind_movimento_permitido', MAX(`ind_movimento_permitido`),
    'ind_item_eliminado', MAX(`ind_item_eliminado`),
    'ind_registro_final', MAX(`ind_registro_final`),
    'ind_item_dummy', MAX(`ind_item_dummy`),
    'ind_material_granel', MAX(`ind_material_granel`),
    'cod_estoque_especial', MAX(`cod_estoque_especial`),
    'cod_centro_custo', MAX(`cod_centro_custo`),
    'cod_elemento_pep', MAX(`cod_elemento_pep`),
    'num_imobilizado', MAX(`num_imobilizado`),
    'num_sub_imobilizado', MAX(`num_sub_imobilizado`),
    'num_diagrama_rede', MAX(`num_diagrama_rede`),
    'num_ordem_cliente', MAX(`num_ordem_cliente`),
    'num_item_ordem_cliente', MAX(`num_item_ordem_cliente`),
    'num_divisao_ordem_cliente', MAX(`num_divisao_ordem_cliente`),
    'desc_observacao', MAX(`desc_observacao`),
    'dateingest', MAX(`dateingest`),
    'yearingest', MAX(`yearingest`),
    'monthingest', MAX(`monthingest`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada.

Ordene pela coluna `veredito`: os problemas aparecem primeiro.

> **Atenção:** nos 5 indicadores `ind_`, o vazio significa **"não"** — é valor legítimo,
> não ausência de dado. O percentual útil deles vai sair baixo sem que haja erro.

In [0]:
-- 6. PREENCHIMENTO DE TODAS AS COLUNAS
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material),
perf AS (
  SELECT stack(49,
    'num_reserva', 'string', COUNT_IF(`num_reserva` IS NULL), COUNT_IF(`num_reserva` IS NOT NULL AND lower(trim(`num_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_reserva`) RLIKE '^0+([.,]0+)?$'),
    'num_item_reserva', 'string', COUNT_IF(`num_item_reserva` IS NULL), COUNT_IF(`num_item_reserva` IS NOT NULL AND lower(trim(`num_item_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_reserva`) RLIKE '^0+([.,]0+)?$'),
    'tp_registro', 'string', COUNT_IF(`tp_registro` IS NULL), COUNT_IF(`tp_registro` IS NOT NULL AND lower(trim(`tp_registro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_registro`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'desc_material', 'string', COUNT_IF(`desc_material` IS NULL), COUNT_IF(`desc_material` IS NOT NULL AND lower(trim(`desc_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_material`) RLIKE '^0+([.,]0+)?$'),
    'num_lote', 'string', COUNT_IF(`num_lote` IS NULL), COUNT_IF(`num_lote` IS NOT NULL AND lower(trim(`num_lote`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_lote`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito', 'string', COUNT_IF(`cod_deposito` IS NULL), COUNT_IF(`cod_deposito` IS NOT NULL AND lower(trim(`cod_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_receptor', 'string', COUNT_IF(`cod_centro_receptor` IS NULL), COUNT_IF(`cod_centro_receptor` IS NOT NULL AND lower(trim(`cod_centro_receptor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_receptor`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito_receptor', 'string', COUNT_IF(`cod_deposito_receptor` IS NULL), COUNT_IF(`cod_deposito_receptor` IS NOT NULL AND lower(trim(`cod_deposito_receptor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito_receptor`) RLIKE '^0+([.,]0+)?$'),
    'dt_necessidade', 'string', COUNT_IF(`dt_necessidade` IS NULL), COUNT_IF(`dt_necessidade` IS NOT NULL AND lower(trim(`dt_necessidade`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_necessidade`) RLIKE '^0+([.,]0+)?$'),
    'dt_reserva', 'string', COUNT_IF(`dt_reserva` IS NULL), COUNT_IF(`dt_reserva` IS NOT NULL AND lower(trim(`dt_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_reserva`) RLIKE '^0+([.,]0+)?$'),
    'dt_criacao', 'string', COUNT_IF(`dt_criacao` IS NULL), COUNT_IF(`dt_criacao` IS NOT NULL AND lower(trim(`dt_criacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_criacao`) RLIKE '^0+([.,]0+)?$'),
    'qt_necessaria', 'decimal(13,3)', COUNT_IF(`qt_necessaria` IS NULL), 0L, COUNT_IF(`qt_necessaria` = 0),
    'qt_retirada', 'decimal(13,3)', COUNT_IF(`qt_retirada` IS NULL), 0L, COUNT_IF(`qt_retirada` = 0),
    'qt_saldo_reserva', 'decimal(14,3)', COUNT_IF(`qt_saldo_reserva` IS NULL), 0L, COUNT_IF(`qt_saldo_reserva` = 0),
    'sg_unidade_medida_basica', 'string', COUNT_IF(`sg_unidade_medida_basica` IS NULL), COUNT_IF(`sg_unidade_medida_basica` IS NOT NULL AND lower(trim(`sg_unidade_medida_basica`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_unidade_medida_basica`) RLIKE '^0+([.,]0+)?$'),
    'qt_um_registro', 'decimal(13,3)', COUNT_IF(`qt_um_registro` IS NULL), 0L, COUNT_IF(`qt_um_registro` = 0),
    'sg_unidade_medida_registro', 'string', COUNT_IF(`sg_unidade_medida_registro` IS NULL), COUNT_IF(`sg_unidade_medida_registro` IS NOT NULL AND lower(trim(`sg_unidade_medida_registro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_unidade_medida_registro`) RLIKE '^0+([.,]0+)?$'),
    'qt_verificacao_disponibilidade', 'decimal(15,3)', COUNT_IF(`qt_verificacao_disponibilidade` IS NULL), 0L, COUNT_IF(`qt_verificacao_disponibilidade` = 0),
    'num_ordem', 'string', COUNT_IF(`num_ordem` IS NULL), COUNT_IF(`num_ordem` IS NOT NULL AND lower(trim(`num_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_ordem`) RLIKE '^0+([.,]0+)?$'),
    'cod_operacao', 'string', COUNT_IF(`cod_operacao` IS NULL), COUNT_IF(`cod_operacao` IS NOT NULL AND lower(trim(`cod_operacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_operacao`) RLIKE '^0+([.,]0+)?$'),
    'tp_movimento', 'string', COUNT_IF(`tp_movimento` IS NULL), COUNT_IF(`tp_movimento` IS NOT NULL AND lower(trim(`tp_movimento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_movimento`) RLIKE '^0+([.,]0+)?$'),
    'tp_necessidade', 'string', COUNT_IF(`tp_necessidade` IS NULL), COUNT_IF(`tp_necessidade` IS NOT NULL AND lower(trim(`tp_necessidade`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_necessidade`) RLIKE '^0+([.,]0+)?$'),
    'cod_debito_credito', 'string', COUNT_IF(`cod_debito_credito` IS NULL), COUNT_IF(`cod_debito_credito` IS NOT NULL AND lower(trim(`cod_debito_credito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_debito_credito`) RLIKE '^0+([.,]0+)?$'),
    'num_ordem_referencia', 'string', COUNT_IF(`num_ordem_referencia` IS NULL), COUNT_IF(`num_ordem_referencia` IS NOT NULL AND lower(trim(`num_ordem_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_ordem_referencia`) RLIKE '^0+([.,]0+)?$'),
    'num_ordem_servico', 'string', COUNT_IF(`num_ordem_servico` IS NULL), COUNT_IF(`num_ordem_servico` IS NOT NULL AND lower(trim(`num_ordem_servico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_ordem_servico`) RLIKE '^0+([.,]0+)?$'),
    'num_item_ordem_servico', 'string', COUNT_IF(`num_item_ordem_servico` IS NULL), COUNT_IF(`num_item_ordem_servico` IS NOT NULL AND lower(trim(`num_item_ordem_servico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_ordem_servico`) RLIKE '^0+([.,]0+)?$'),
    'st_reserva', 'string', COUNT_IF(`st_reserva` IS NULL), COUNT_IF(`st_reserva` IS NOT NULL AND lower(trim(`st_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_reserva`) RLIKE '^0+([.,]0+)?$'),
    'ind_movimento_permitido', 'string', COUNT_IF(`ind_movimento_permitido` IS NULL), COUNT_IF(`ind_movimento_permitido` IS NOT NULL AND lower(trim(`ind_movimento_permitido`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_movimento_permitido`) RLIKE '^0+([.,]0+)?$'),
    'ind_item_eliminado', 'string', COUNT_IF(`ind_item_eliminado` IS NULL), COUNT_IF(`ind_item_eliminado` IS NOT NULL AND lower(trim(`ind_item_eliminado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_item_eliminado`) RLIKE '^0+([.,]0+)?$'),
    'ind_registro_final', 'string', COUNT_IF(`ind_registro_final` IS NULL), COUNT_IF(`ind_registro_final` IS NOT NULL AND lower(trim(`ind_registro_final`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_registro_final`) RLIKE '^0+([.,]0+)?$'),
    'ind_item_dummy', 'string', COUNT_IF(`ind_item_dummy` IS NULL), COUNT_IF(`ind_item_dummy` IS NOT NULL AND lower(trim(`ind_item_dummy`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_item_dummy`) RLIKE '^0+([.,]0+)?$'),
    'ind_material_granel', 'string', COUNT_IF(`ind_material_granel` IS NULL), COUNT_IF(`ind_material_granel` IS NOT NULL AND lower(trim(`ind_material_granel`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_material_granel`) RLIKE '^0+([.,]0+)?$'),
    'cod_estoque_especial', 'string', COUNT_IF(`cod_estoque_especial` IS NULL), COUNT_IF(`cod_estoque_especial` IS NOT NULL AND lower(trim(`cod_estoque_especial`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_estoque_especial`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_custo', 'string', COUNT_IF(`cod_centro_custo` IS NULL), COUNT_IF(`cod_centro_custo` IS NOT NULL AND lower(trim(`cod_centro_custo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_custo`) RLIKE '^0+([.,]0+)?$'),
    'cod_elemento_pep', 'string', COUNT_IF(`cod_elemento_pep` IS NULL), COUNT_IF(`cod_elemento_pep` IS NOT NULL AND lower(trim(`cod_elemento_pep`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_elemento_pep`) RLIKE '^0+([.,]0+)?$'),
    'num_imobilizado', 'string', COUNT_IF(`num_imobilizado` IS NULL), COUNT_IF(`num_imobilizado` IS NOT NULL AND lower(trim(`num_imobilizado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_imobilizado`) RLIKE '^0+([.,]0+)?$'),
    'num_sub_imobilizado', 'string', COUNT_IF(`num_sub_imobilizado` IS NULL), COUNT_IF(`num_sub_imobilizado` IS NOT NULL AND lower(trim(`num_sub_imobilizado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_sub_imobilizado`) RLIKE '^0+([.,]0+)?$'),
    'num_diagrama_rede', 'string', COUNT_IF(`num_diagrama_rede` IS NULL), COUNT_IF(`num_diagrama_rede` IS NOT NULL AND lower(trim(`num_diagrama_rede`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_diagrama_rede`) RLIKE '^0+([.,]0+)?$'),
    'num_ordem_cliente', 'string', COUNT_IF(`num_ordem_cliente` IS NULL), COUNT_IF(`num_ordem_cliente` IS NOT NULL AND lower(trim(`num_ordem_cliente`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_ordem_cliente`) RLIKE '^0+([.,]0+)?$'),
    'num_item_ordem_cliente', 'string', COUNT_IF(`num_item_ordem_cliente` IS NULL), COUNT_IF(`num_item_ordem_cliente` IS NOT NULL AND lower(trim(`num_item_ordem_cliente`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_ordem_cliente`) RLIKE '^0+([.,]0+)?$'),
    'num_divisao_ordem_cliente', 'string', COUNT_IF(`num_divisao_ordem_cliente` IS NULL), COUNT_IF(`num_divisao_ordem_cliente` IS NOT NULL AND lower(trim(`num_divisao_ordem_cliente`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_divisao_ordem_cliente`) RLIKE '^0+([.,]0+)?$'),
    'nm_usuario', 'string', COUNT_IF(`nm_usuario` IS NULL), COUNT_IF(`nm_usuario` IS NOT NULL AND lower(trim(`nm_usuario`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_usuario`) RLIKE '^0+([.,]0+)?$'),
    'nm_recebedor_mercadoria', 'string', COUNT_IF(`nm_recebedor_mercadoria` IS NULL), COUNT_IF(`nm_recebedor_mercadoria` IS NOT NULL AND lower(trim(`nm_recebedor_mercadoria`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_recebedor_mercadoria`) RLIKE '^0+([.,]0+)?$'),
    'desc_observacao', 'string', COUNT_IF(`desc_observacao` IS NULL), COUNT_IF(`desc_observacao` IS NOT NULL AND lower(trim(`desc_observacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_observacao`) RLIKE '^0+([.,]0+)?$'),
    'dateingest', 'date', COUNT_IF(`dateingest` IS NULL), 0L, 0L,
    'yearingest', 'string', COUNT_IF(`yearingest` IS NULL), COUNT_IF(`yearingest` IS NOT NULL AND lower(trim(`yearingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`yearingest`) RLIKE '^0+([.,]0+)?$'),
    'monthingest', 'string', COUNT_IF(`monthingest` IS NULL), COUNT_IF(`monthingest` IS NOT NULL AND lower(trim(`monthingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`monthingest`) RLIKE '^0+([.,]0+)?$')
  ) AS (coluna, tipo, nulos, vazios, zeros)
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros,
       t.total - p.nulos - p.vazios - p.zeros                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                       THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                              AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

Coluna constante é candidata a valor default de carga.

In [0]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material),
card AS (
  SELECT stack(49,
    'num_reserva', 'string', approx_count_distinct(`num_reserva`),
    'num_item_reserva', 'string', approx_count_distinct(`num_item_reserva`),
    'tp_registro', 'string', approx_count_distinct(`tp_registro`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'desc_material', 'string', approx_count_distinct(`desc_material`),
    'num_lote', 'string', approx_count_distinct(`num_lote`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_deposito', 'string', approx_count_distinct(`cod_deposito`),
    'cod_centro_receptor', 'string', approx_count_distinct(`cod_centro_receptor`),
    'cod_deposito_receptor', 'string', approx_count_distinct(`cod_deposito_receptor`),
    'dt_necessidade', 'string', approx_count_distinct(`dt_necessidade`),
    'dt_reserva', 'string', approx_count_distinct(`dt_reserva`),
    'dt_criacao', 'string', approx_count_distinct(`dt_criacao`),
    'qt_necessaria', 'decimal(13,3)', approx_count_distinct(`qt_necessaria`),
    'qt_retirada', 'decimal(13,3)', approx_count_distinct(`qt_retirada`),
    'qt_saldo_reserva', 'decimal(14,3)', approx_count_distinct(`qt_saldo_reserva`),
    'sg_unidade_medida_basica', 'string', approx_count_distinct(`sg_unidade_medida_basica`),
    'qt_um_registro', 'decimal(13,3)', approx_count_distinct(`qt_um_registro`),
    'sg_unidade_medida_registro', 'string', approx_count_distinct(`sg_unidade_medida_registro`),
    'qt_verificacao_disponibilidade', 'decimal(15,3)', approx_count_distinct(`qt_verificacao_disponibilidade`),
    'num_ordem', 'string', approx_count_distinct(`num_ordem`),
    'cod_operacao', 'string', approx_count_distinct(`cod_operacao`),
    'tp_movimento', 'string', approx_count_distinct(`tp_movimento`),
    'tp_necessidade', 'string', approx_count_distinct(`tp_necessidade`),
    'cod_debito_credito', 'string', approx_count_distinct(`cod_debito_credito`),
    'num_ordem_referencia', 'string', approx_count_distinct(`num_ordem_referencia`),
    'num_ordem_servico', 'string', approx_count_distinct(`num_ordem_servico`),
    'num_item_ordem_servico', 'string', approx_count_distinct(`num_item_ordem_servico`),
    'st_reserva', 'string', approx_count_distinct(`st_reserva`),
    'ind_movimento_permitido', 'string', approx_count_distinct(`ind_movimento_permitido`),
    'ind_item_eliminado', 'string', approx_count_distinct(`ind_item_eliminado`),
    'ind_registro_final', 'string', approx_count_distinct(`ind_registro_final`),
    'ind_item_dummy', 'string', approx_count_distinct(`ind_item_dummy`),
    'ind_material_granel', 'string', approx_count_distinct(`ind_material_granel`),
    'cod_estoque_especial', 'string', approx_count_distinct(`cod_estoque_especial`),
    'cod_centro_custo', 'string', approx_count_distinct(`cod_centro_custo`),
    'cod_elemento_pep', 'string', approx_count_distinct(`cod_elemento_pep`),
    'num_imobilizado', 'string', approx_count_distinct(`num_imobilizado`),
    'num_sub_imobilizado', 'string', approx_count_distinct(`num_sub_imobilizado`),
    'num_diagrama_rede', 'string', approx_count_distinct(`num_diagrama_rede`),
    'num_ordem_cliente', 'string', approx_count_distinct(`num_ordem_cliente`),
    'num_item_ordem_cliente', 'string', approx_count_distinct(`num_item_ordem_cliente`),
    'num_divisao_ordem_cliente', 'string', approx_count_distinct(`num_divisao_ordem_cliente`),
    'nm_usuario', 'string', approx_count_distinct(`nm_usuario`),
    'nm_recebedor_mercadoria', 'string', approx_count_distinct(`nm_recebedor_mercadoria`),
    'desc_observacao', 'string', approx_count_distinct(`desc_observacao`),
    'dateingest', 'date', approx_count_distinct(`dateingest`),
    'yearingest', 'string', approx_count_distinct(`yearingest`),
    'monthingest', 'string', approx_count_distinct(`monthingest`)
  ) AS (coluna, tipo, distintos)
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma.

> 🔒 `nm_usuario` e `nm_recebedor_mercadoria` excluídas — dados pessoais.

In [0]:
-- 8. DOMINIO DAS CATEGORICAS
(SELECT 'tp_registro' AS coluna, CAST(`tp_registro` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `tp_registro` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_centro' AS coluna, CAST(`cod_centro` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `cod_centro` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_deposito' AS coluna, CAST(`cod_deposito` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `cod_deposito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_movimento' AS coluna, CAST(`tp_movimento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `tp_movimento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_necessidade' AS coluna, CAST(`tp_necessidade` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `tp_necessidade` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_debito_credito' AS coluna, CAST(`cod_debito_credito` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `cod_debito_credito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_reserva' AS coluna, CAST(`st_reserva` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `st_reserva` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_estoque_especial' AS coluna, CAST(`cod_estoque_especial` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `cod_estoque_especial` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_unidade_medida_basica' AS coluna, CAST(`sg_unidade_medida_basica` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `sg_unidade_medida_basica` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_unidade_medida_registro' AS coluna, CAST(`sg_unidade_medida_registro` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `sg_unidade_medida_registro` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_movimento_permitido' AS coluna, CAST(`ind_movimento_permitido` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `ind_movimento_permitido` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_item_eliminado' AS coluna, CAST(`ind_item_eliminado` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `ind_item_eliminado` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_registro_final' AS coluna, CAST(`ind_registro_final` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `ind_registro_final` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_item_dummy' AS coluna, CAST(`ind_item_dummy` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `ind_item_dummy` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_material_granel' AS coluna, CAST(`ind_material_granel` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material GROUP BY `ind_material_granel` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Indicadores `X` / vazio

Os 5 indicadores estão em STRING com domínio `X` / vazio — **mesmo formato do SAP**,
então a comparação é direta. Qualquer valor em `fora_do_dominio` é anomalia.

In [0]:
-- 9. INDICADORES
SELECT 'ind_movimento_permitido' AS indicador,
       COUNT_IF(trim(COALESCE(`ind_movimento_permitido`,'')) = 'X') AS marcado_X,
       COUNT_IF(trim(COALESCE(`ind_movimento_permitido`,'')) = '')  AS vazio,
       COUNT_IF(trim(COALESCE(`ind_movimento_permitido`,'')) NOT IN ('X', '')) AS fora_do_dominio,
       ROUND(100.0 * COUNT_IF(trim(COALESCE(`ind_movimento_permitido`,'')) = 'X') / COUNT(*), 2) AS pct_X
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
UNION ALL
SELECT 'ind_item_eliminado' AS indicador,
       COUNT_IF(trim(COALESCE(`ind_item_eliminado`,'')) = 'X') AS marcado_X,
       COUNT_IF(trim(COALESCE(`ind_item_eliminado`,'')) = '')  AS vazio,
       COUNT_IF(trim(COALESCE(`ind_item_eliminado`,'')) NOT IN ('X', '')) AS fora_do_dominio,
       ROUND(100.0 * COUNT_IF(trim(COALESCE(`ind_item_eliminado`,'')) = 'X') / COUNT(*), 2) AS pct_X
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
UNION ALL
SELECT 'ind_registro_final' AS indicador,
       COUNT_IF(trim(COALESCE(`ind_registro_final`,'')) = 'X') AS marcado_X,
       COUNT_IF(trim(COALESCE(`ind_registro_final`,'')) = '')  AS vazio,
       COUNT_IF(trim(COALESCE(`ind_registro_final`,'')) NOT IN ('X', '')) AS fora_do_dominio,
       ROUND(100.0 * COUNT_IF(trim(COALESCE(`ind_registro_final`,'')) = 'X') / COUNT(*), 2) AS pct_X
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
UNION ALL
SELECT 'ind_item_dummy' AS indicador,
       COUNT_IF(trim(COALESCE(`ind_item_dummy`,'')) = 'X') AS marcado_X,
       COUNT_IF(trim(COALESCE(`ind_item_dummy`,'')) = '')  AS vazio,
       COUNT_IF(trim(COALESCE(`ind_item_dummy`,'')) NOT IN ('X', '')) AS fora_do_dominio,
       ROUND(100.0 * COUNT_IF(trim(COALESCE(`ind_item_dummy`,'')) = 'X') / COUNT(*), 2) AS pct_X
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
UNION ALL
SELECT 'ind_material_granel' AS indicador,
       COUNT_IF(trim(COALESCE(`ind_material_granel`,'')) = 'X') AS marcado_X,
       COUNT_IF(trim(COALESCE(`ind_material_granel`,'')) = '')  AS vazio,
       COUNT_IF(trim(COALESCE(`ind_material_granel`,'')) NOT IN ('X', '')) AS fora_do_dominio,
       ROUND(100.0 * COUNT_IF(trim(COALESCE(`ind_material_granel`,'')) = 'X') / COUNT(*), 2) AS pct_X
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
ORDER BY pct_X DESC;

## 10. Perfil dos campos numéricos

Todos os 5 campos numéricos são `decimal` — **exatos**, seguros para conciliação sem tolerância.

In [0]:
-- 10. PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(5,
    'qt_necessaria', 'decimal(13,3)', COUNT(`qt_necessaria`), CAST(MIN(`qt_necessaria`) AS DOUBLE), CAST(MAX(`qt_necessaria`) AS DOUBLE), CAST(AVG(`qt_necessaria`) AS DOUBLE), CAST(percentile_approx(`qt_necessaria`, 0.5) AS DOUBLE), COUNT_IF(`qt_necessaria` < 0), COUNT_IF(`qt_necessaria` = 0),
    'qt_retirada', 'decimal(13,3)', COUNT(`qt_retirada`), CAST(MIN(`qt_retirada`) AS DOUBLE), CAST(MAX(`qt_retirada`) AS DOUBLE), CAST(AVG(`qt_retirada`) AS DOUBLE), CAST(percentile_approx(`qt_retirada`, 0.5) AS DOUBLE), COUNT_IF(`qt_retirada` < 0), COUNT_IF(`qt_retirada` = 0),
    'qt_saldo_reserva', 'decimal(14,3)', COUNT(`qt_saldo_reserva`), CAST(MIN(`qt_saldo_reserva`) AS DOUBLE), CAST(MAX(`qt_saldo_reserva`) AS DOUBLE), CAST(AVG(`qt_saldo_reserva`) AS DOUBLE), CAST(percentile_approx(`qt_saldo_reserva`, 0.5) AS DOUBLE), COUNT_IF(`qt_saldo_reserva` < 0), COUNT_IF(`qt_saldo_reserva` = 0),
    'qt_um_registro', 'decimal(13,3)', COUNT(`qt_um_registro`), CAST(MIN(`qt_um_registro`) AS DOUBLE), CAST(MAX(`qt_um_registro`) AS DOUBLE), CAST(AVG(`qt_um_registro`) AS DOUBLE), CAST(percentile_approx(`qt_um_registro`, 0.5) AS DOUBLE), COUNT_IF(`qt_um_registro` < 0), COUNT_IF(`qt_um_registro` = 0),
    'qt_verificacao_disponibilidade', 'decimal(15,3)', COUNT(`qt_verificacao_disponibilidade`), CAST(MIN(`qt_verificacao_disponibilidade`) AS DOUBLE), CAST(MAX(`qt_verificacao_disponibilidade`) AS DOUBLE), CAST(AVG(`qt_verificacao_disponibilidade`) AS DOUBLE), CAST(percentile_approx(`qt_verificacao_disponibilidade`, 0.5) AS DOUBLE), COUNT_IF(`qt_verificacao_disponibilidade` < 0), COUNT_IF(`qt_verificacao_disponibilidade` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
)
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato da MB25.**

Atenção: `qt_necessaria`, `qt_retirada` e `qt_saldo_reserva` estão na **unidade básica**;
`qt_um_registro` está na **unidade de registro**. Some no Excel a coluna correspondente.

In [0]:
-- 11. TOTAIS PARA CONCILIACAO
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(5,
    'qt_necessaria', CAST(SUM(`qt_necessaria`) AS DOUBLE), CAST(ROUND(SUM(`qt_necessaria`), 3) AS STRING), COUNT(`qt_necessaria`),
    'qt_retirada', CAST(SUM(`qt_retirada`) AS DOUBLE), CAST(ROUND(SUM(`qt_retirada`), 3) AS STRING), COUNT(`qt_retirada`),
    'qt_saldo_reserva', CAST(SUM(`qt_saldo_reserva`) AS DOUBLE), CAST(ROUND(SUM(`qt_saldo_reserva`), 3) AS STRING), COUNT(`qt_saldo_reserva`),
    'qt_um_registro', CAST(SUM(`qt_um_registro`) AS DOUBLE), CAST(ROUND(SUM(`qt_um_registro`), 3) AS STRING), COUNT(`qt_um_registro`),
    'qt_verificacao_disponibilidade', CAST(SUM(`qt_verificacao_disponibilidade`) AS DOUBLE), CAST(ROUND(SUM(`qt_verificacao_disponibilidade`), 3) AS STRING), COUNT(`qt_verificacao_disponibilidade`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
)
ORDER BY coluna;

## 12. Datas armazenadas como STRING

**Armadilha:** o SAP exporta `2024-02-23 00:00:00` e o Datalake pode gravar `20240223`.
Mesma data, formato diferente — já gerou 16.773 falsos positivos em validação anterior.

In [0]:
-- 12. DATAS EM STRING
SELECT coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, data_zero, minimo, maximo,
       CASE WHEN (CASE WHEN fmt_AAAAMMDD > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_ISO       > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_BR        > 0 THEN 1 ELSE 0 END) > 1
            THEN 'ALERTA: mais de um formato' ELSE 'formato unico' END AS veredito
FROM (
  SELECT stack(3,
    'dt_necessidade', COUNT_IF(`dt_necessidade` IS NULL OR trim(`dt_necessidade`) = ''), COUNT_IF(trim(`dt_necessidade`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_necessidade`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_necessidade`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_necessidade`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_necessidade`) NOT IN ('', '00000000') THEN `dt_necessidade` END), MAX(CASE WHEN trim(`dt_necessidade`) NOT IN ('', '00000000') THEN `dt_necessidade` END),
    'dt_reserva', COUNT_IF(`dt_reserva` IS NULL OR trim(`dt_reserva`) = ''), COUNT_IF(trim(`dt_reserva`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_reserva`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_reserva`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_reserva`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_reserva`) NOT IN ('', '00000000') THEN `dt_reserva` END), MAX(CASE WHEN trim(`dt_reserva`) NOT IN ('', '00000000') THEN `dt_reserva` END),
    'dt_criacao', COUNT_IF(`dt_criacao` IS NULL OR trim(`dt_criacao`) = ''), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_criacao`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_criacao`) NOT IN ('', '00000000') THEN `dt_criacao` END), MAX(CASE WHEN trim(`dt_criacao`) NOT IN ('', '00000000') THEN `dt_criacao` END)
  ) AS (coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, data_zero, minimo, maximo)
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
)
ORDER BY coluna;

In [0]:
-- 12.1 DATA DE INGESTAO (tipo nativo)
SELECT COUNT_IF(dateingest IS NULL)        AS nulos,
       MIN(dateingest)                     AS primeira,
       MAX(dateingest)                     AS ultima,
       COUNT(DISTINCT dateingest)          AS distintas
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material;

## 13. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake pode gravar `000000000000425263`.
Sem normalizar, o join dá 0% de match.

In [0]:
-- 13. CODIGOS
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(14,
    'num_reserva', 'string', COUNT_IF(CAST(`num_reserva` AS STRING) IS NULL OR trim(CAST(`num_reserva` AS STRING)) = ''), MIN(length(trim(CAST(`num_reserva` AS STRING)))), MAX(length(trim(CAST(`num_reserva` AS STRING)))), COUNT_IF(trim(CAST(`num_reserva` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_reserva` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_reserva` AS STRING)), '^0+', '')),
    'num_item_reserva', 'string', COUNT_IF(CAST(`num_item_reserva` AS STRING) IS NULL OR trim(CAST(`num_item_reserva` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_reserva` AS STRING)))), MAX(length(trim(CAST(`num_item_reserva` AS STRING)))), COUNT_IF(trim(CAST(`num_item_reserva` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_reserva` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_reserva` AS STRING)), '^0+', '')),
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '')),
    'cod_deposito', 'string', COUNT_IF(CAST(`cod_deposito` AS STRING) IS NULL OR trim(CAST(`cod_deposito` AS STRING)) = ''), MIN(length(trim(CAST(`cod_deposito` AS STRING)))), MAX(length(trim(CAST(`cod_deposito` AS STRING)))), COUNT_IF(trim(CAST(`cod_deposito` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_deposito` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_deposito` AS STRING)), '^0+', '')),
    'cod_centro_receptor', 'string', COUNT_IF(CAST(`cod_centro_receptor` AS STRING) IS NULL OR trim(CAST(`cod_centro_receptor` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro_receptor` AS STRING)))), MAX(length(trim(CAST(`cod_centro_receptor` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro_receptor` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro_receptor` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro_receptor` AS STRING)), '^0+', '')),
    'cod_deposito_receptor', 'string', COUNT_IF(CAST(`cod_deposito_receptor` AS STRING) IS NULL OR trim(CAST(`cod_deposito_receptor` AS STRING)) = ''), MIN(length(trim(CAST(`cod_deposito_receptor` AS STRING)))), MAX(length(trim(CAST(`cod_deposito_receptor` AS STRING)))), COUNT_IF(trim(CAST(`cod_deposito_receptor` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_deposito_receptor` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_deposito_receptor` AS STRING)), '^0+', '')),
    'num_ordem', 'string', COUNT_IF(CAST(`num_ordem` AS STRING) IS NULL OR trim(CAST(`num_ordem` AS STRING)) = ''), MIN(length(trim(CAST(`num_ordem` AS STRING)))), MAX(length(trim(CAST(`num_ordem` AS STRING)))), COUNT_IF(trim(CAST(`num_ordem` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_ordem` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_ordem` AS STRING)), '^0+', '')),
    'num_ordem_referencia', 'string', COUNT_IF(CAST(`num_ordem_referencia` AS STRING) IS NULL OR trim(CAST(`num_ordem_referencia` AS STRING)) = ''), MIN(length(trim(CAST(`num_ordem_referencia` AS STRING)))), MAX(length(trim(CAST(`num_ordem_referencia` AS STRING)))), COUNT_IF(trim(CAST(`num_ordem_referencia` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_ordem_referencia` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_ordem_referencia` AS STRING)), '^0+', '')),
    'num_ordem_servico', 'string', COUNT_IF(CAST(`num_ordem_servico` AS STRING) IS NULL OR trim(CAST(`num_ordem_servico` AS STRING)) = ''), MIN(length(trim(CAST(`num_ordem_servico` AS STRING)))), MAX(length(trim(CAST(`num_ordem_servico` AS STRING)))), COUNT_IF(trim(CAST(`num_ordem_servico` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_ordem_servico` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_ordem_servico` AS STRING)), '^0+', '')),
    'cod_centro_custo', 'string', COUNT_IF(CAST(`cod_centro_custo` AS STRING) IS NULL OR trim(CAST(`cod_centro_custo` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro_custo` AS STRING)))), MAX(length(trim(CAST(`cod_centro_custo` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro_custo` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro_custo` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro_custo` AS STRING)), '^0+', '')),
    'cod_elemento_pep', 'string', COUNT_IF(CAST(`cod_elemento_pep` AS STRING) IS NULL OR trim(CAST(`cod_elemento_pep` AS STRING)) = ''), MIN(length(trim(CAST(`cod_elemento_pep` AS STRING)))), MAX(length(trim(CAST(`cod_elemento_pep` AS STRING)))), COUNT_IF(trim(CAST(`cod_elemento_pep` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_elemento_pep` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_elemento_pep` AS STRING)), '^0+', '')),
    'num_imobilizado', 'string', COUNT_IF(CAST(`num_imobilizado` AS STRING) IS NULL OR trim(CAST(`num_imobilizado` AS STRING)) = ''), MIN(length(trim(CAST(`num_imobilizado` AS STRING)))), MAX(length(trim(CAST(`num_imobilizado` AS STRING)))), COUNT_IF(trim(CAST(`num_imobilizado` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_imobilizado` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_imobilizado` AS STRING)), '^0+', '')),
    'num_lote', 'string', COUNT_IF(CAST(`num_lote` AS STRING) IS NULL OR trim(CAST(`num_lote` AS STRING)) = ''), MIN(length(trim(CAST(`num_lote` AS STRING)))), MAX(length(trim(CAST(`num_lote` AS STRING)))), COUNT_IF(trim(CAST(`num_lote` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_lote` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_lote` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
)
ORDER BY coluna;

## 14. Checksum de linha

Se `linhas` > `linhas_unicas`, existem registros 100% idênticos — duplicata real.

In [0]:
-- 14. CHECKSUM DE LINHA
SELECT COUNT(*)                                            AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_reserva` AS STRING), ''), COALESCE(CAST(`num_item_reserva` AS STRING), ''), COALESCE(CAST(`tp_registro` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`num_lote` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`cod_centro_receptor` AS STRING), ''), COALESCE(CAST(`cod_deposito_receptor` AS STRING), ''), COALESCE(CAST(`dt_necessidade` AS STRING), ''), COALESCE(CAST(`dt_reserva` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`qt_necessaria` AS STRING), ''), COALESCE(CAST(`qt_retirada` AS STRING), ''), COALESCE(CAST(`qt_saldo_reserva` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`qt_um_registro` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_registro` AS STRING), ''), COALESCE(CAST(`qt_verificacao_disponibilidade` AS STRING), ''), COALESCE(CAST(`num_ordem` AS STRING), ''), COALESCE(CAST(`cod_operacao` AS STRING), ''), COALESCE(CAST(`tp_movimento` AS STRING), ''), COALESCE(CAST(`tp_necessidade` AS STRING), ''), COALESCE(CAST(`cod_debito_credito` AS STRING), ''), COALESCE(CAST(`num_ordem_referencia` AS STRING), ''), COALESCE(CAST(`num_ordem_servico` AS STRING), ''), COALESCE(CAST(`num_item_ordem_servico` AS STRING), ''), COALESCE(CAST(`st_reserva` AS STRING), ''), COALESCE(CAST(`ind_movimento_permitido` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado` AS STRING), ''), COALESCE(CAST(`ind_registro_final` AS STRING), ''), COALESCE(CAST(`ind_item_dummy` AS STRING), ''), COALESCE(CAST(`ind_material_granel` AS STRING), ''), COALESCE(CAST(`cod_estoque_especial` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`num_imobilizado` AS STRING), ''), COALESCE(CAST(`num_sub_imobilizado` AS STRING), ''), COALESCE(CAST(`num_diagrama_rede` AS STRING), ''), COALESCE(CAST(`num_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_item_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_divisao_ordem_cliente` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`nm_recebedor_mercadoria` AS STRING), ''), COALESCE(CAST(`desc_observacao` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), ''))))            AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_reserva` AS STRING), ''), COALESCE(CAST(`num_item_reserva` AS STRING), ''), COALESCE(CAST(`tp_registro` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`num_lote` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`cod_centro_receptor` AS STRING), ''), COALESCE(CAST(`cod_deposito_receptor` AS STRING), ''), COALESCE(CAST(`dt_necessidade` AS STRING), ''), COALESCE(CAST(`dt_reserva` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`qt_necessaria` AS STRING), ''), COALESCE(CAST(`qt_retirada` AS STRING), ''), COALESCE(CAST(`qt_saldo_reserva` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`qt_um_registro` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_registro` AS STRING), ''), COALESCE(CAST(`qt_verificacao_disponibilidade` AS STRING), ''), COALESCE(CAST(`num_ordem` AS STRING), ''), COALESCE(CAST(`cod_operacao` AS STRING), ''), COALESCE(CAST(`tp_movimento` AS STRING), ''), COALESCE(CAST(`tp_necessidade` AS STRING), ''), COALESCE(CAST(`cod_debito_credito` AS STRING), ''), COALESCE(CAST(`num_ordem_referencia` AS STRING), ''), COALESCE(CAST(`num_ordem_servico` AS STRING), ''), COALESCE(CAST(`num_item_ordem_servico` AS STRING), ''), COALESCE(CAST(`st_reserva` AS STRING), ''), COALESCE(CAST(`ind_movimento_permitido` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado` AS STRING), ''), COALESCE(CAST(`ind_registro_final` AS STRING), ''), COALESCE(CAST(`ind_item_dummy` AS STRING), ''), COALESCE(CAST(`ind_material_granel` AS STRING), ''), COALESCE(CAST(`cod_estoque_especial` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`num_imobilizado` AS STRING), ''), COALESCE(CAST(`num_sub_imobilizado` AS STRING), ''), COALESCE(CAST(`num_diagrama_rede` AS STRING), ''), COALESCE(CAST(`num_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_item_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_divisao_ordem_cliente` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`nm_recebedor_mercadoria` AS STRING), ''), COALESCE(CAST(`desc_observacao` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), '')))) AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_reserva` AS STRING), ''), COALESCE(CAST(`num_item_reserva` AS STRING), ''), COALESCE(CAST(`tp_registro` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`num_lote` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`cod_centro_receptor` AS STRING), ''), COALESCE(CAST(`cod_deposito_receptor` AS STRING), ''), COALESCE(CAST(`dt_necessidade` AS STRING), ''), COALESCE(CAST(`dt_reserva` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`qt_necessaria` AS STRING), ''), COALESCE(CAST(`qt_retirada` AS STRING), ''), COALESCE(CAST(`qt_saldo_reserva` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`qt_um_registro` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_registro` AS STRING), ''), COALESCE(CAST(`qt_verificacao_disponibilidade` AS STRING), ''), COALESCE(CAST(`num_ordem` AS STRING), ''), COALESCE(CAST(`cod_operacao` AS STRING), ''), COALESCE(CAST(`tp_movimento` AS STRING), ''), COALESCE(CAST(`tp_necessidade` AS STRING), ''), COALESCE(CAST(`cod_debito_credito` AS STRING), ''), COALESCE(CAST(`num_ordem_referencia` AS STRING), ''), COALESCE(CAST(`num_ordem_servico` AS STRING), ''), COALESCE(CAST(`num_item_ordem_servico` AS STRING), ''), COALESCE(CAST(`st_reserva` AS STRING), ''), COALESCE(CAST(`ind_movimento_permitido` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado` AS STRING), ''), COALESCE(CAST(`ind_registro_final` AS STRING), ''), COALESCE(CAST(`ind_item_dummy` AS STRING), ''), COALESCE(CAST(`ind_material_granel` AS STRING), ''), COALESCE(CAST(`cod_estoque_especial` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`num_imobilizado` AS STRING), ''), COALESCE(CAST(`num_sub_imobilizado` AS STRING), ''), COALESCE(CAST(`num_diagrama_rede` AS STRING), ''), COALESCE(CAST(`num_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_item_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_divisao_ordem_cliente` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`nm_recebedor_mercadoria` AS STRING), ''), COALESCE(CAST(`desc_observacao` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material;

## 15. Amostra de linhas completas

> 🔒 `nm_usuario` e `nm_recebedor_mercadoria` aparecem **mascaradas por hash**.

In [0]:
-- 15. AMOSTRA (dados pessoais mascarados)
SELECT `num_reserva`,
       `num_item_reserva`,
       `tp_registro`,
       `cod_material`,
       `desc_material`,
       `num_lote`,
       `cod_centro`,
       `cod_deposito`,
       `cod_centro_receptor`,
       `cod_deposito_receptor`,
       `dt_necessidade`,
       `dt_reserva`,
       `dt_criacao`,
       `qt_necessaria`,
       `qt_retirada`,
       `qt_saldo_reserva`,
       `sg_unidade_medida_basica`,
       `qt_um_registro`,
       `sg_unidade_medida_registro`,
       `qt_verificacao_disponibilidade`,
       `num_ordem`,
       `cod_operacao`,
       `tp_movimento`,
       `tp_necessidade`,
       `cod_debito_credito`,
       `num_ordem_referencia`,
       `num_ordem_servico`,
       `num_item_ordem_servico`,
       `st_reserva`,
       `ind_movimento_permitido`,
       `ind_item_eliminado`,
       `ind_registro_final`,
       `ind_item_dummy`,
       `ind_material_granel`,
       `cod_estoque_especial`,
       `cod_centro_custo`,
       `cod_elemento_pep`,
       `num_imobilizado`,
       `num_sub_imobilizado`,
       `num_diagrama_rede`,
       `num_ordem_cliente`,
       `num_item_ordem_cliente`,
       `num_divisao_ordem_cliente`,
       substring(md5(`nm_usuario`), 1, 8) AS `nm_usuario_hash`,
       substring(md5(`nm_recebedor_mercadoria`), 1, 8) AS `nm_recebedor_mercadoria_hash`,
       `desc_observacao`,
       `dateingest`,
       `yearingest`,
       `monthingest`
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
ORDER BY `num_reserva`, `num_item_reserva`
LIMIT 20;

## 16. Distribuição interna

Como o volume se reparte nas principais dimensões.

In [0]:
-- 16. DISTRIBUICAO POR cod_centro
SELECT COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)') AS `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material), 2) AS pct
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR tp_movimento
SELECT COALESCE(NULLIF(trim(CAST(`tp_movimento` AS STRING)), ''), '(vazio)') AS `tp_movimento`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material), 2) AS pct
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_movimento` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR tp_necessidade
SELECT COALESCE(NULLIF(trim(CAST(`tp_necessidade` AS STRING)), ''), '(vazio)') AS `tp_necessidade`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material), 2) AS pct
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_necessidade` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR st_reserva
SELECT COALESCE(NULLIF(trim(CAST(`st_reserva` AS STRING)), ''), '(vazio)') AS `st_reserva`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material), 2) AS pct
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
GROUP BY COALESCE(NULLIF(trim(CAST(`st_reserva` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_deposito
SELECT COALESCE(NULLIF(trim(CAST(`cod_deposito` AS STRING)), ''), '(vazio)') AS `cod_deposito`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material), 2) AS pct
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_deposito` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

## 17. Freshness

In [0]:
-- 17. FRESHNESS
SELECT dateingest AS data_carga, COUNT(*) AS linhas, COUNT(DISTINCT num_reserva) AS reservas
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
GROUP BY dateingest
ORDER BY data_carga DESC
LIMIT 30;

In [0]:
-- 17.1 SNAPSHOT OU HISTORICO?
SELECT MIN(dateingest)            AS primeira_carga,
       MAX(dateingest)            AS ultima_carga,
       COUNT(DISTINCT dateingest) AS cargas_distintas,
       CASE WHEN COUNT(DISTINCT dateingest) = 1
            THEN 'SNAPSHOT - substitui a cada carga; chave NAO precisa da data'
            ELSE 'HISTORICO - acumula; a chave DEVE incluir dateingest' END AS veredito
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material;

## 18. Análises específicas — MB25

### 18.1 Validação de `qt_saldo_reserva`

O schema declara: `qt_saldo_reserva = qt_necessaria − qt_retirada` (calculado).
Esta célula confere a fórmula em todas as linhas.

In [0]:
-- 18.1 VALIDACAO DO SALDO CALCULADO
SELECT COUNT(*)                                                             AS linhas,
       COUNT_IF(qt_saldo_reserva = qt_necessaria - qt_retirada)             AS formula_confere,
       COUNT_IF(qt_saldo_reserva <> qt_necessaria - qt_retirada)            AS formula_diverge,
       COUNT_IF(qt_saldo_reserva IS NULL)                                   AS saldo_nulo,
       MAX(ABS(qt_saldo_reserva - (qt_necessaria - qt_retirada)))           AS maior_diferenca,
       CASE WHEN COUNT_IF(qt_saldo_reserva <> qt_necessaria - qt_retirada) = 0
            THEN 'OK - formula confere' ELSE 'DIVERGE - investigar' END     AS veredito
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material;

### 18.2 Necessário × retirado

Retirada maior que o necessário é possível no SAP (retirada a maior), mas precisa ser conhecida.
Saldo negativo é consequência direta.

In [0]:
-- 18.2 NECESSARIO x RETIRADO
SELECT COUNT(*)                                            AS linhas,
       COUNT_IF(qt_retirada = 0)                           AS nada_retirado,
       COUNT_IF(qt_retirada > 0 AND qt_retirada < qt_necessaria) AS retirada_parcial,
       COUNT_IF(qt_retirada = qt_necessaria)               AS totalmente_atendida,
       COUNT_IF(qt_retirada > qt_necessaria)               AS retirada_a_maior,
       COUNT_IF(qt_saldo_reserva < 0)                      AS saldo_negativo,
       COUNT_IF(qt_necessaria <= 0)                        AS necessaria_nao_positiva
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material;

In [0]:
-- 18.2b ATENDIMENTO x REGISTRO FINAL
SELECT COALESCE(NULLIF(trim(ind_registro_final), ''), '(vazio)') AS registro_final,
       CASE WHEN qt_retirada = 0 THEN 'nada retirado'
            WHEN qt_retirada >= qt_necessaria THEN 'total'
            ELSE 'parcial' END AS atendimento,
       COUNT(*) AS linhas
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
GROUP BY COALESCE(NULLIF(trim(ind_registro_final), ''), '(vazio)'),
         CASE WHEN qt_retirada = 0 THEN 'nada retirado'
              WHEN qt_retirada >= qt_necessaria THEN 'total'
              ELSE 'parcial' END
ORDER BY linhas DESC;

### 18.3 🔴 Campos de cabeçalho replicados nos itens

O schema declara que 7 campos vêm do **cabeçalho** da reserva e foram **replicados em todos
os itens** — mas na RESB (tabela SAP de itens) eles são campos de item.

**Consequência:** se no SAP um item tiver valor próprio diferente do cabeçalho, a view mostra
o valor do cabeçalho e a comparação diverge. Esta célula confirma a replicação:
dentro de cada reserva, o campo deve ter **um único valor**.

In [0]:
-- 18.3 REPLICACAO DE CAMPOS DE CABECALHO
SELECT 'num_imobilizado' AS campo,
       COUNT(*) AS reservas_com_valor,
       COUNT_IF(valores > 1) AS reservas_com_mais_de_um_valor,
       CASE WHEN COUNT_IF(valores > 1) = 0 THEN 'REPLICACAO CONFIRMADA'
            ELSE 'ha variacao entre itens' END AS veredito
  FROM (SELECT num_reserva, COUNT(DISTINCT `num_imobilizado`) AS valores
          FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
         WHERE trim(COALESCE(`num_imobilizado`,'')) <> ''
         GROUP BY num_reserva)
UNION ALL
SELECT 'num_sub_imobilizado' AS campo,
       COUNT(*) AS reservas_com_valor,
       COUNT_IF(valores > 1) AS reservas_com_mais_de_um_valor,
       CASE WHEN COUNT_IF(valores > 1) = 0 THEN 'REPLICACAO CONFIRMADA'
            ELSE 'ha variacao entre itens' END AS veredito
  FROM (SELECT num_reserva, COUNT(DISTINCT `num_sub_imobilizado`) AS valores
          FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
         WHERE trim(COALESCE(`num_sub_imobilizado`,'')) <> ''
         GROUP BY num_reserva)
UNION ALL
SELECT 'num_diagrama_rede' AS campo,
       COUNT(*) AS reservas_com_valor,
       COUNT_IF(valores > 1) AS reservas_com_mais_de_um_valor,
       CASE WHEN COUNT_IF(valores > 1) = 0 THEN 'REPLICACAO CONFIRMADA'
            ELSE 'ha variacao entre itens' END AS veredito
  FROM (SELECT num_reserva, COUNT(DISTINCT `num_diagrama_rede`) AS valores
          FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
         WHERE trim(COALESCE(`num_diagrama_rede`,'')) <> ''
         GROUP BY num_reserva)
UNION ALL
SELECT 'num_ordem_cliente' AS campo,
       COUNT(*) AS reservas_com_valor,
       COUNT_IF(valores > 1) AS reservas_com_mais_de_um_valor,
       CASE WHEN COUNT_IF(valores > 1) = 0 THEN 'REPLICACAO CONFIRMADA'
            ELSE 'ha variacao entre itens' END AS veredito
  FROM (SELECT num_reserva, COUNT(DISTINCT `num_ordem_cliente`) AS valores
          FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
         WHERE trim(COALESCE(`num_ordem_cliente`,'')) <> ''
         GROUP BY num_reserva)
UNION ALL
SELECT 'num_item_ordem_cliente' AS campo,
       COUNT(*) AS reservas_com_valor,
       COUNT_IF(valores > 1) AS reservas_com_mais_de_um_valor,
       CASE WHEN COUNT_IF(valores > 1) = 0 THEN 'REPLICACAO CONFIRMADA'
            ELSE 'ha variacao entre itens' END AS veredito
  FROM (SELECT num_reserva, COUNT(DISTINCT `num_item_ordem_cliente`) AS valores
          FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
         WHERE trim(COALESCE(`num_item_ordem_cliente`,'')) <> ''
         GROUP BY num_reserva)
UNION ALL
SELECT 'num_divisao_ordem_cliente' AS campo,
       COUNT(*) AS reservas_com_valor,
       COUNT_IF(valores > 1) AS reservas_com_mais_de_um_valor,
       CASE WHEN COUNT_IF(valores > 1) = 0 THEN 'REPLICACAO CONFIRMADA'
            ELSE 'ha variacao entre itens' END AS veredito
  FROM (SELECT num_reserva, COUNT(DISTINCT `num_divisao_ordem_cliente`) AS valores
          FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
         WHERE trim(COALESCE(`num_divisao_ordem_cliente`,'')) <> ''
         GROUP BY num_reserva)
UNION ALL
SELECT 'cod_centro_custo' AS campo,
       COUNT(*) AS reservas_com_valor,
       COUNT_IF(valores > 1) AS reservas_com_mais_de_um_valor,
       CASE WHEN COUNT_IF(valores > 1) = 0 THEN 'REPLICACAO CONFIRMADA'
            ELSE 'ha variacao entre itens' END AS veredito
  FROM (SELECT num_reserva, COUNT(DISTINCT `cod_centro_custo`) AS valores
          FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
         WHERE trim(COALESCE(`cod_centro_custo`,'')) <> ''
         GROUP BY num_reserva);

### 18.4 Tipo de movimento × débito/crédito × objeto de imputação

`num_ordem_referencia` é declarado como objeto de imputação para **tipo de movimento 261**
(saída para ordem). Esta célula verifica a coerência.

In [0]:
-- 18.4 MOVIMENTO x DEBITO/CREDITO x IMPUTACAO
SELECT COALESCE(NULLIF(trim(tp_movimento), ''), '(vazio)')       AS tp_movimento,
       COALESCE(NULLIF(trim(cod_debito_credito), ''), '(vazio)') AS debito_credito,
       COUNT(*)                                                   AS linhas,
       COUNT_IF(trim(COALESCE(num_ordem,'')) <> '')               AS com_ordem,
       COUNT_IF(trim(COALESCE(num_ordem_referencia,'')) <> '')    AS com_ordem_referencia,
       COUNT_IF(trim(COALESCE(cod_centro_custo,'')) <> '')        AS com_centro_custo,
       COUNT_IF(trim(COALESCE(cod_centro_receptor,'')) <> '')     AS com_centro_receptor
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
GROUP BY COALESCE(NULLIF(trim(tp_movimento), ''), '(vazio)'),
         COALESCE(NULLIF(trim(cod_debito_credito), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 30;

In [0]:
-- 18.4b COMBINACOES SUSPEITAS
SELECT 'movimento 261 SEM ordem de referencia' AS situacao,
       COUNT_IF(trim(tp_movimento) = '261'
            AND trim(COALESCE(num_ordem_referencia,'')) = '') AS linhas
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
UNION ALL
SELECT 'transferencia (3xx) SEM centro receptor',
       COUNT_IF(trim(tp_movimento) LIKE '3%'
            AND trim(COALESCE(cod_centro_receptor,'')) = '')
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
UNION ALL
SELECT 'item eliminado MAS com retirada',
       COUNT_IF(trim(COALESCE(ind_item_eliminado,'')) = 'X' AND qt_retirada > 0)
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
UNION ALL
SELECT 'movimento nao permitido MAS com retirada',
       COUNT_IF(trim(COALESCE(ind_movimento_permitido,'')) = '' AND qt_retirada > 0)
  FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
ORDER BY linhas DESC;

### 18.5 Unidade básica × unidade de registro

`qt_necessaria` está na unidade básica; `qt_um_registro` na unidade de registro.
Quando as duas unidades são iguais, as quantidades deveriam ser iguais.

In [0]:
-- 18.5 UNIDADES DE MEDIDA
SELECT COUNT(*)                                                         AS linhas,
       COUNT_IF(sg_unidade_medida_basica = sg_unidade_medida_registro)  AS mesma_unidade,
       COUNT_IF(sg_unidade_medida_basica = sg_unidade_medida_registro
            AND qt_necessaria <> qt_um_registro)                        AS mesma_unidade_qtd_diferente,
       COUNT_IF(sg_unidade_medida_basica <> sg_unidade_medida_registro) AS unidades_diferentes,
       COUNT_IF(qt_verificacao_disponibilidade > qt_necessaria)         AS atp_maior_que_necessario
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material;

### 18.6 Coerência de datas

Sequência esperada: data da reserva ≤ data de necessidade.

In [0]:
-- 18.6 COERENCIA DE DATAS
SELECT COUNT(*)                                                    AS linhas,
       COUNT_IF(COALESCE(to_date(CASE WHEN trim(`dt_reserva`) RLIKE '^[0-9]{8}$' AND trim(`dt_reserva`) <> '00000000' THEN trim(`dt_reserva`) END, 'yyyyMMdd'), to_date(`dt_reserva`)) IS NULL)                        AS reserva_nao_convertida,
       COUNT_IF(COALESCE(to_date(CASE WHEN trim(`dt_necessidade`) RLIKE '^[0-9]{8}$' AND trim(`dt_necessidade`) <> '00000000' THEN trim(`dt_necessidade`) END, 'yyyyMMdd'), to_date(`dt_necessidade`)) IS NULL)                    AS necessidade_nao_convertida,
       COUNT_IF(COALESCE(to_date(CASE WHEN trim(`dt_reserva`) RLIKE '^[0-9]{8}$' AND trim(`dt_reserva`) <> '00000000' THEN trim(`dt_reserva`) END, 'yyyyMMdd'), to_date(`dt_reserva`)) > COALESCE(to_date(CASE WHEN trim(`dt_necessidade`) RLIKE '^[0-9]{8}$' AND trim(`dt_necessidade`) <> '00000000' THEN trim(`dt_necessidade`) END, 'yyyyMMdd'), to_date(`dt_necessidade`)))       AS reserva_apos_necessidade,
       MIN(COALESCE(to_date(CASE WHEN trim(`dt_reserva`) RLIKE '^[0-9]{8}$' AND trim(`dt_reserva`) <> '00000000' THEN trim(`dt_reserva`) END, 'yyyyMMdd'), to_date(`dt_reserva`)))                                     AS reserva_min,
       MAX(COALESCE(to_date(CASE WHEN trim(`dt_reserva`) RLIKE '^[0-9]{8}$' AND trim(`dt_reserva`) <> '00000000' THEN trim(`dt_reserva`) END, 'yyyyMMdd'), to_date(`dt_reserva`)))                                     AS reserva_max,
       MIN(COALESCE(to_date(CASE WHEN trim(`dt_necessidade`) RLIKE '^[0-9]{8}$' AND trim(`dt_necessidade`) <> '00000000' THEN trim(`dt_necessidade`) END, 'yyyyMMdd'), to_date(`dt_necessidade`)))                                 AS necessidade_min,
       MAX(COALESCE(to_date(CASE WHEN trim(`dt_necessidade`) RLIKE '^[0-9]{8}$' AND trim(`dt_necessidade`) <> '00000000' THEN trim(`dt_necessidade`) END, 'yyyyMMdd'), to_date(`dt_necessidade`)))                                 AS necessidade_max
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material;

### 18.7 🔒 Dados pessoais — somente agregados

In [0]:
-- 18.7 USUARIOS E RECEBEDORES (somente contagens)
SELECT COUNT(DISTINCT nm_usuario)                                AS usuarios_distintos,
       COUNT_IF(trim(COALESCE(nm_usuario,'')) = '')              AS sem_usuario,
       COUNT(DISTINCT nm_recebedor_mercadoria)                   AS recebedores_distintos,
       COUNT_IF(trim(COALESCE(nm_recebedor_mercadoria,'')) = '') AS sem_recebedor
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material;

## 19. Amostra ampliada

O notebook exploratório não extrai a base completa. Para isso, use um notebook de cenário
com o recorte escolhido na seção 3.

In [0]:
-- 19. AMOSTRA AMPLIADA (dados pessoais mascarados)
SELECT `num_reserva`,
       `num_item_reserva`,
       `tp_registro`,
       `cod_material`,
       `desc_material`,
       `num_lote`,
       `cod_centro`,
       `cod_deposito`,
       `cod_centro_receptor`,
       `cod_deposito_receptor`,
       `dt_necessidade`,
       `dt_reserva`,
       `dt_criacao`,
       `qt_necessaria`,
       `qt_retirada`,
       `qt_saldo_reserva`,
       `sg_unidade_medida_basica`,
       `qt_um_registro`,
       `sg_unidade_medida_registro`,
       `qt_verificacao_disponibilidade`,
       `num_ordem`,
       `cod_operacao`,
       `tp_movimento`,
       `tp_necessidade`,
       `cod_debito_credito`,
       `num_ordem_referencia`,
       `num_ordem_servico`,
       `num_item_ordem_servico`,
       `st_reserva`,
       `ind_movimento_permitido`,
       `ind_item_eliminado`,
       `ind_registro_final`,
       `ind_item_dummy`,
       `ind_material_granel`,
       `cod_estoque_especial`,
       `cod_centro_custo`,
       `cod_elemento_pep`,
       `num_imobilizado`,
       `num_sub_imobilizado`,
       `num_diagrama_rede`,
       `num_ordem_cliente`,
       `num_item_ordem_cliente`,
       `num_divisao_ordem_cliente`,
       substring(md5(`nm_usuario`), 1, 8) AS `nm_usuario_hash`,
       substring(md5(`nm_recebedor_mercadoria`), 1, 8) AS `nm_recebedor_mercadoria_hash`,
       `desc_observacao`,
       `dateingest`,
       `yearingest`,
       `monthingest`
FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material
ORDER BY `num_reserva`, `num_item_reserva`
LIMIT 500;

## 20. Resumo

**Copie esta saída** e envie ao agente junto com o notebook.

In [0]:
-- 20. RESUMO
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material),
g AS (
  SELECT 'num_reserva' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_reserva` FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material)
UNION ALL
  SELECT 'num_reserva + num_item_reserva' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva` FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material)
UNION ALL
  SELECT 'num_reserva + num_item_reserva + tp_registro' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva`, `tp_registro` FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material)
UNION ALL
  SELECT 'num_reserva + num_item_reserva + tp_registro + dateingest' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva`, `tp_registro`, `dateingest` FROM prd_procurement.corp_curated.vw_ds_log_mb25_reservas_material)
)
SELECT 'IDENTIFICACAO' AS bloco, 'transacao' AS item, 'MB25' AS valor
UNION ALL SELECT 'IDENTIFICACAO', 'ambiente', 'PRD'
UNION ALL SELECT 'IDENTIFICACAO', 'tabela', 'vw_ds_log_mb25_reservas_material'
UNION ALL SELECT 'IDENTIFICACAO', 'tipo de objeto', 'VIEW'
UNION ALL SELECT 'IDENTIFICACAO', 'linhas', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'IDENTIFICACAO', 'colunas', '49'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Escolher o recorte de teste com base na **seção 3** (conferir 4128 e 4014 na seção 3.1).
2. Gerar os notebooks de cenário com esse recorte.
3. Extrair a MB25 no **SAP PRD** com o mesmo recorte e na **mesma data** do snapshot.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Checklist antes de comparar

- [ ] Chave real identificada (seção 4)
- [ ] `qt_saldo_reserva` validado (seção 18.1)
- [ ] Campos de cabeçalho replicados conferidos (seção 18.3)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 13)
- [ ] Formato de data normalizado (seção 12)
- [ ] Totais numéricos conferidos na unidade correta (seção 11)
- [ ] 🔒 Dados pessoais tratados só por agregado
